# Analytics Basics

This notebook covers analytical SQL patterns:
- Window functions
- Common Table Expressions (CTEs)
- Advanced aggregations
- Subqueries

In [ ]:
from databend_driver import connect

conn = connect("databend+local:///./local-state")

## Setup: Create sample data

In [ ]:
conn.execute("DROP TABLE IF EXISTS sales")
conn.execute("""
    CREATE TABLE sales (
        id INT,
        product VARCHAR,
        region VARCHAR,
        amount INT,
        sale_date DATE
    )
""")
conn.execute("""
    INSERT INTO sales VALUES
    (1, 'Laptop', 'North', 1200, '2024-01-15'),
    (2, 'Phone', 'North', 800, '2024-01-20'),
    (3, 'Laptop', 'South', 1100, '2024-02-10'),
    (4, 'Tablet', 'North', 500, '2024-02-15'),
    (5, 'Phone', 'South', 750, '2024-03-01'),
    (6, 'Laptop', 'North', 1300, '2024-03-10'),
    (7, 'Tablet', 'South', 450, '2024-03-15'),
    (8, 'Phone', 'North', 850, '2024-04-01')
""")
print("Sales data loaded!")

## 1. Window Functions

Window functions perform calculations across a set of rows related to the current row.

In [ ]:
# Rank sales within each region
result = conn.query("""
    SELECT product, region, amount,
           RANK() OVER (PARTITION BY region ORDER BY amount DESC) AS rank
    FROM sales
""")
print(result.fetchall())

In [ ]:
# Running total per region
result = conn.query("""
    SELECT product, region, amount,
           SUM(amount) OVER (PARTITION BY region ORDER BY sale_date) AS running_total
    FROM sales
""")
print(result.fetchall())

## 2. Common Table Expressions (CTEs)

CTEs make complex queries more readable by breaking them into named steps.

In [ ]:
# Find products that sell above average in each region
result = conn.query("""
    WITH region_avg AS (
        SELECT region, AVG(amount) AS avg_amount
        FROM sales
        GROUP BY region
    )
    SELECT s.product, s.region, s.amount, r.avg_amount
    FROM sales s
    JOIN region_avg r ON s.region = r.region
    WHERE s.amount > r.avg_amount
""")
print(result.fetchall())

## 3. Advanced Aggregations

In [ ]:
# Monthly sales summary withROLLUP
result = conn.query("""
    SELECT 
        EXTRACT(MONTH FROM sale_date) AS month,
        product,
        SUM(amount) AS total
    FROM sales
    GROUP BY ROLLUP(month, product)
    ORDER BY month, product
""")
print(result.fetchall())

In [ ]:
# Percentage of total per region
result = conn.query("""
    SELECT product, region, amount,
           ROUND(amount * 100.0 / SUM(amount) OVER (PARTITION BY region), 1) AS pct
    FROM sales
""")
print(result.fetchall())

## 4. Subqueries

In [ ]:
# Products with above-average total sales
result = conn.query("""
    SELECT product, SUM(amount) AS total
    FROM sales
    GROUP BY product
    HAVING total > (SELECT AVG(amount) * 3 FROM sales)
""")
print(result.fetchall())